# Proyecto del Día 15 - Optimizar Modelos de Machine Learning

El proyecto del día de hoy consiste en tomar un código que ya está desarrollado con Scikit-Learn, y modificarlo agregando varios apspectos que hemos aprendido el día de hoy.

Este programa es un simple análisis usando el modelo de **Bosques Aleatorios** en un dataset de **Scikit-Learn** sobre registros médicos de **pacientes con diabetes**.

### Consigna

Tu trabajo aquí simplemente consiste en ejecutar este código, y analizarlo para identificar qué hace cada línea, y cuál es su función dentro del código. Una vez que te hayas asegurado de haber identificado sus elementos y de haber comprendido su funcionamiento en general, realiza todas las modificaciones que creas necesarias, para que este código incluya:

+ Preprocesamiento de datos con StandardScaler
+ Selección de mejores categorías (investiga cuál es la mejor función de cálculo)
+ Pipelines
+ Evaluación de modelo

Suena fácil, pero te aseguro que poner todo junto en un mismo proyecto, puede hacer que las cosas se compliquen un poco.

Te deseo lo mejor como siempre, esperando que este proyecto implique un buen tiempo de diversión y resolución de problemas.

In [17]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Cargar el dataset de diabetes
diabetes = datasets.load_diabetes()

# Convertir a DataFrame para facilitar el análisis exploratorio
diabetes_df = pd.DataFrame(data=np.c_[diabetes['data'], diabetes['target']],
                           columns=diabetes['feature_names'] + ['target'])

# Convertir 'target' en categorías para clasificación
diabetes_df['target'] = (diabetes_df['target'] > diabetes_df['target'].median()).astype(int)

# División de datos en conjuntos de entrenamiento y prueba
X = diabetes_df.drop('target', axis=1)
y = diabetes_df['target']
X_entrena, X_prueba, y_entrena, y_prueba = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear y entrenar el modelo RandomForest
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_entrena, y_entrena)

# Realizar predicciones con el conjunto de prueba
predicciones = modelo.predict(X_prueba)

# Evaluación del modelo
puntaje = modelo.score(X_prueba, y_prueba)
print(f"\nPrecisión del modelo: {puntaje:.2f}")



Precisión del modelo: 0.72


### Solución

In [12]:
# Codigo para validar el valor de K optimo en SelectKBest como selector 
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Cargar dataset
diabetes = datasets.load_diabetes()

# Convertir a DataFrame
diabetes_df = pd.DataFrame(
    data=np.c_[diabetes['data'], diabetes['target']],
    columns=diabetes['feature_names'] + ['target']
)

# Convertir target en binario
diabetes_df['target'] = (diabetes_df['target'] > diabetes_df['target'].median()).astype(int)

# Separar variables
X = diabetes_df.drop('target', axis=1)
y = diabetes_df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Probar distintos valores de k
for k in range(1, X.shape[1] + 1):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SelectKBest(score_func=f_classif, k=k)), #Se utilizó f_classif porque el target fue transformado en clasificación binaria, y esta función permite medir la relación estadística entre cada característica y la clase objetivo
        ('model', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"k={k} -> accuracy={acc:.3f}")

k=1 -> accuracy=0.739
k=2 -> accuracy=0.721
k=3 -> accuracy=0.739
k=4 -> accuracy=0.775
k=5 -> accuracy=0.775
k=6 -> accuracy=0.775
k=7 -> accuracy=0.757
k=8 -> accuracy=0.748
k=9 -> accuracy=0.748
k=10 -> accuracy=0.748


SelectKBest tiene valor optimo en K=4

####      CODIGO OPTIMIZADO

In [19]:
# Importamos las librerías necesarias
import numpy as np
import pandas as pd

# Importamos el dataset de diabetes desde scikit-learn
from sklearn import datasets

# Importamos herramientas para dividir los datos
from sklearn.model_selection import train_test_split, cross_val_score

# Importamos Pipeline para encadenar pasos de preprocesamiento y modelado
from sklearn.pipeline import Pipeline

# Importamos StandardScaler para estandarizar las variables
from sklearn.preprocessing import StandardScaler

# Importamos SelectKBest para seleccionar las mejores variables
# y f_classif porque el problema es de clasificación
from sklearn.feature_selection import SelectKBest, f_classif

# Importamos el modelo de clasificación
from sklearn.ensemble import RandomForestClassifier

# Importamos métricas para evaluar el modelo
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Cargamos el dataset de diabetes
diabetes = datasets.load_diabetes()

# Convertimos el dataset en un DataFrame para trabajar más fácilmente
diabetes_df = pd.DataFrame(
    data=np.c_[diabetes['data'], diabetes['target']],
    columns=diabetes['feature_names'] + ['target']
)

# Convertimos el target en una variable binaria:
# si el valor es mayor que la mediana, asignamos 1; si no, 0
diabetes_df['target'] = (diabetes_df['target'] > diabetes_df['target'].median()).astype(int)

# Separamos variables predictoras y variable objetivo
X = diabetes_df.drop('target', axis=1)
y = diabetes_df['target']

# Dividimos los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Creamos el pipeline con tres pasos:
# 1. Escalado de datos
# 2. Selección de las mejores variables
# 3. Entrenamiento del modelo
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif, k=4)),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Entrenamos el pipeline con los datos de entrenamiento
pipeline.fit(X_train, y_train)

# Realizamos predicciones sobre el conjunto de prueba
y_pred = pipeline.predict(X_test)

# Calculamos la exactitud del modelo
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Mostramos la matriz de confusión
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

# Mostramos el reporte de clasificación
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))

# Validación cruzada sobre todo el dataset
scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')

print("\nValidación cruzada (accuracy por fold):")
print(scores)

print(f"\nPromedio de validación cruzada: {scores.mean():.2f}")
print(f"Desviación estándar: {scores.std():.2f}")

Accuracy: 0.74

Matriz de confusión:
[[35 14]
 [ 9 31]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.80      0.71      0.75        49
           1       0.69      0.78      0.73        40

    accuracy                           0.74        89
   macro avg       0.74      0.74      0.74        89
weighted avg       0.75      0.74      0.74        89


Validación cruzada (accuracy por fold):
[0.69662921 0.74157303 0.67045455 0.67045455 0.68181818]

Promedio de validación cruzada: 0.69
Desviación estándar: 0.03
